In [1]:
import os

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

## Understanding Dot Product and Cosine Similarity between Word Embeddings

### **1. Introduction**

In modern Natural Language Processing (NLP), words, phrases, or sentences are often represented as **vectors** — called **embeddings**.
These embeddings capture **semantic meaning**, so that words with similar meanings lie close to each other in a high-dimensional space.

For example, words like *“King”* and *“Queen”* should be closer in vector space than *“King”* and *“Table.”*

---

### **2. The Role of the Dot Product**

The **dot product** measures how much two vectors point in the **same direction**.
For vectors **A** and **B**, it’s defined as:

$$
A \cdot B = |A|, |B| \cos(\theta)
$$

* ( |A| ) and ( |B| ) are the **magnitudes (lengths)** of the vectors.
* ( \theta ) is the **angle** between them.

If the dot product is **large and positive**, the vectors point in **similar directions**, meaning the words are **semantically related**.
If it’s **near zero**, the words are **unrelated**.
If it’s **negative**, they are **oppositely related** (rare in word embeddings).

---

### **3. Cosine Similarity**

The **cosine similarity** normalizes the dot product to remove the effect of vector length.
It focuses **only on the angle** between vectors and is defined as:

$$
\text{Cosine Similarity} = \frac{A \cdot B}{|A|,|B|}
$$

* **Range:** −1 to 1

  * `1` → perfectly aligned (meanings are nearly identical)
  * `0` → unrelated
  * `−1` → completely opposite

In word embeddings:

* **High cosine similarity (close to 1)** → words have **similar meanings**.
* **Low cosine similarity (close to 0)** → words are **semantically distant**.

---

### **4. Example: “King” and “Queen”**

When you encode:

```python
question = "King"
document = "Queen"
```

and calculate their **cosine similarity**, the resulting score (e.g., `0.85`) means:

* The two vectors are **strongly aligned**, forming a **small angle** between them.
* Hence, the model understands that *“King”* and *“Queen”* are **semantically related** — both represent **royalty** but differ in gender.

If you convert the cosine similarity to an **angle**, smaller angles imply greater semantic closeness:

$$
\theta = \cos^{-1}(0.85) \approx 31.8^\circ
$$

---

### **5. Why It Matters**

Understanding dot product and cosine similarity helps you:

* **Quantify semantic similarity** between words or sentences.
* **Compare relationships** (e.g., *“King–Queen” ≈ “Man–Woman”*).
* **Build systems** like semantic search, question answering, or RAG models that rely on **contextual similarity**.

---

### **6. Summary**

| Concept               | Purpose                                    | Output Type      | Interpretation                       |
| --------------------- | ------------------------------------------ | ---------------- | ------------------------------------ |
| **Dot Product**       | Measures directional alignment + magnitude | Scalar           | Larger value = more aligned          |
| **Cosine Similarity** | Measures pure directional alignment        | Scalar (−1 to 1) | Higher = more semantically similar   |
| **Application**       | NLP embeddings                             | —                | Reveals how related two meanings are |

---

So, the topic **“Understanding Dot Product and Cosine Similarity between Word Embeddings”** means exploring how mathematical relationships between vectors — through dot product and cosine similarity — help models detect **semantic relationships** between words in embedding space.

In [4]:
# Encode both the question and document into dense vector representations
question = "King"
document = "Queen"

# Convert text into numerical embeddings (high-dimensional vectors)
question_vect = model.encode(question)
docs_vect = model.encode(document)

# Compute the dot product between the two embeddings
# This measures how semantically similar the two sentences are
similarity_score = np.dot(question_vect, docs_vect)

# Display results
print("=== Embedding Similarity Analysis ===")
print(f"Question: {question}")
print(f"Document: {document}")
print(f"\nQuestion Vector (first 5 values): {question_vect[:5]}")
print(f"Document Vector (first 5 values): {docs_vect[:5]}")

print("\nFormula: Similarity = |v1| × |v2| × cos(θ)")
print(f"Dot Product (Similarity Score): {similarity_score:.6f}")

# Optional: If you want to show the cosine similarity explicitly
norm_question = np.linalg.norm(question_vect)
norm_docs = np.linalg.norm(docs_vect)
cosine_similarity = similarity_score / (norm_question * norm_docs)
angle_radians = np.arccos(cosine_similarity)
angle_degrees = np.degrees(angle_radians)
print(f"Cosine Similarity (Normalized): {cosine_similarity:.6f}")
print("\nConclusion:")
print(f"This means the 'question vector' and 'document vector' are nearly aligned, forming an angle of approximately 0.589328 radians (≈ {angle_degrees:.2f}°) between them — indicating that they are semantically close in meaning.")

=== Embedding Similarity Analysis ===
Question: King
Document: Queen

Question Vector (first 5 values): [-0.05959932  0.05051239 -0.06951008  0.07968022 -0.0467477 ]
Document Vector (first 5 values): [ 0.03548699 -0.06560468 -0.00993496  0.03159032 -0.01338684]

Formula: Similarity = |v1| × |v2| × cos(θ)
Dot Product (Similarity Score): 0.680713
Cosine Similarity (Normalized): 0.680713

Conclusion:
This means the 'question vector' and 'document vector' are nearly aligned, forming an angle of approximately 0.589328 radians (≈ 47.10°) between them — indicating that they are semantically close in meaning.


## Measuring Semantic and Relational Similarity using Embedding Vectors
1. **`calculate_similarity_score`**

   * Computes how semantically close two words or phrases are.
   * Uses cosine similarity as the metric.

2. **Standard deviation among scores**

   * Checks if the model treats all gender/relational pairs similarly.
   * Smaller values → more consistency.

3. **Difference vectors (`get_difference_vect`)**

   * Represent relationships rather than meanings.
   * Example: “King − Queen” vector captures a “male-to-female royalty” direction.

4. **Pairwise comparison**

   * Measures whether all such relationships (King–Queen, Male–Female, Men–Women) point in a similar direction.

5. **Conclusion section**

   * Summarizes what the results imply about your embedding model’s understanding of relationships.

In [5]:
import numpy as np

# Function to calculate cosine similarity between two text inputs
def calculate_similarity_score(model, question, docs):
    # Encode both inputs into embedding vectors
    question_vect = model.encode(question)
    docs_vect = model.encode(docs)
    
    # Calculate their magnitudes (vector lengths)
    question_vect_magnitude = np.linalg.norm(question_vect)
    docs_vect_magnitude = np.linalg.norm(docs_vect)

    # Compute the dot product and then derive cosine similarity
    dot_product = np.dot(question_vect, docs_vect)
    cosine_similarity = dot_product / (question_vect_magnitude * docs_vect_magnitude)

    return cosine_similarity


# Comparing how semantically close certain word pairs are
men_women_similarity_score = calculate_similarity_score(model, "Men", "Women")
king_queen_similarity_score = calculate_similarity_score(model, "King", "Queen")
male_female_similarity_score = calculate_similarity_score(model, "Male", "Female")

# Store and analyze the variation between these similarity scores
similarity_scores = [men_women_similarity_score, king_queen_similarity_score, male_female_similarity_score]
similarity_std = np.std(similarity_scores)

print("=== Semantic Similarity Results ===")
print(f"Men ↔ Women Similarity: {men_women_similarity_score:.6f}")
print(f"King ↔ Queen Similarity: {king_queen_similarity_score:.6f}")
print(f"Male ↔ Female Similarity: {male_female_similarity_score:.6f}")
print(f"Standard Deviation between these similarities: {similarity_std:.6f}")
print("Lower standard deviation indicates that the model captures gender or relational patterns consistently.\n")


# Function to find the difference vector between two related words
def get_difference_vect(model, word1, word2):
    word1_vect = model.encode(word1)
    word2_vect = model.encode(word2)
    return word1_vect - word2_vect


# Function to calculate cosine similarity between two difference vectors
def calculate_cosine_similarity_score_vect(v1, v2):
    dot_product = np.dot(v1, v2)
    v1_magnitude = np.linalg.norm(v1)
    v2_magnitude = np.linalg.norm(v2)
    cosine_similarity = dot_product / (v1_magnitude * v2_magnitude)
    return cosine_similarity


# Generate difference vectors (relationship direction in vector space)
kq = get_difference_vect(model, "King", "Queen")
mf = get_difference_vect(model, "Male", "Female")
mw = get_difference_vect(model, "Men", "Women")

# Store all relationship vectors
vectors = [kq, mf, mw]
vector_names = ["King–Queen", "Male–Female", "Men–Women"]

# Compute pairwise cosine similarities between relationship vectors
print("=== Relationship Vector Similarity Matrix ===")
cosine_similarities = []
for i, v1 in enumerate(vectors):
    for j, v2 in enumerate(vectors):
        score = calculate_cosine_similarity_score_vect(v1, v2)
        cosine_similarities.append(score)
        print(f"{vector_names[i]} ↔ {vector_names[j]}: {score:.6f}")

# Analyze the variation across all pairwise similarity scores
relationship_std = np.std(np.array(cosine_similarities))
print(f"\nStandard Deviation of all relationship similarities: {relationship_std:.6f}")

print("\nConclusion:")
print("If the pairwise similarities are high and the standard deviation is low, "
      "it suggests that the model captures consistent relational patterns between these word pairs. "
      "In this context, the vectors representing 'King–Queen', 'Male–Female', and 'Men–Women' "
      "point in nearly the same semantic direction, showing that the model understands "
      "the analogy or gender-based relationship between them.")

=== Semantic Similarity Results ===
Men ↔ Women Similarity: 0.719090
King ↔ Queen Similarity: 0.680713
Male ↔ Female Similarity: 0.733793
Standard Deviation between these similarities: 0.022377
Lower standard deviation indicates that the model captures gender or relational patterns consistently.

=== Relationship Vector Similarity Matrix ===
King–Queen ↔ King–Queen: 1.000000
King–Queen ↔ Male–Female: 0.378769
King–Queen ↔ Men–Women: 0.331722
Male–Female ↔ King–Queen: 0.378769
Male–Female ↔ Male–Female: 1.000000
Male–Female ↔ Men–Women: 0.790963
Men–Women ↔ King–Queen: 0.331722
Men–Women ↔ Male–Female: 0.790963
Men–Women ↔ Men–Women: 1.000000

Standard Deviation of all relationship similarities: 0.289516

Conclusion:
If the pairwise similarities are high and the standard deviation is low, it suggests that the model captures consistent relational patterns between these word pairs. In this context, the vectors representing 'King–Queen', 'Male–Female', and 'Men–Women' point in nearly the s

## **Core RAG Architecture**

The **core architecture of a Retrieval-Augmented Generation (RAG) system** follows a three-stage pipeline designed to bridge the gap between *retrieval-based factual accuracy* and *generative reasoning capability*.

#### **1. Indexing Stage**

In this phase, external knowledge sources such as documents, PDFs, or databases are processed and transformed into searchable vector representations.

* The text corpus is **chunked** into smaller segments.
* Each segment is **encoded into embeddings** using a transformer-based model (e.g., Sentence-BERT, Gemini).
* The resulting vectors are **stored in a vector database** (e.g., FAISS, Chroma, Pinecone) for efficient similarity search.

#### **2. Retrieval Stage**

When a user query is received:

* The query is **encoded** into its vector form.
* A **similarity search** (commonly cosine similarity or dot product) is performed against the indexed vectors.
* The top-*k* most relevant chunks are **retrieved** as contextual evidence.

#### **3. Generation Stage**

The retrieved context is combined with the original query and passed to a **language model (LLM)** to generate a context-grounded response.
This ensures the final answer is:

* **Factual**, as it references retrieved data.
* **Contextually relevant**, aligning with the user’s intent.
* **Coherent**, leveraging the generative strength of the LLM.

---

### **Formula Representation**

$$
\text{RAG}(q) = \text{LLM}(q, \text{Retrieve}(q, D))
$$

Where:

* ( q ): user query
* ( D ): external document corpus
* Retrieve(q, D): top-k documents retrieved based on similarity
* LLM(q,Retrieve(q, D)): final grounded response

---

This **Index–Retrieve–Generate** pipeline forms the foundation of all modern RAG systems.
Advanced variants (e.g., Self-RAG, Query Construction, or Routing) build upon this by adding **feedback loops**, **query reformulation**, and **adaptive retrieval strategies**.

# Step-1: Indexing

This section focuses on the **Indexing** process, the foundation of any Retrieval-Augmented Generation (RAG) system.
It involves **segmenting textual data into smaller, semantically consistent chunks** and **encoding them into dense vector representations** using an embedding model.
These embeddings serve as the searchable knowledge base, enabling efficient and meaningful retrieval during the generation phase.
This structured approach ensures that large text corpora can be accessed and queried with high semantic precision.

In [6]:
# Import necessary class
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Create a text splitter with defined chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 50,    # Each chunk will contain around 10 tokens
    chunk_overlap = 3   # 3 tokens of overlap between consecutive chunks
)

def load_documents():
    """
    Loads a small, diverse set of documents to simulate a knowledge base
    for Retrieval-Augmented Generation (RAG) experiments.

    Each document represents a single knowledge unit that will later be
    chunked, embedded, and indexed for similarity-based retrieval.

    Returns
    -------
    documents : list of langchain_core.documents.Document
        A list containing 10 textual documents across multiple domains.
    """

    # Define a diverse mini knowledge base covering various topics
    documents = [
        Document(page_content="The Eiffel Tower is located in Paris, France, and stands at 324 meters tall."),
        Document(page_content="The Great Wall of China stretches over 13,000 miles and was built to prevent invasions."),
        Document(page_content="Machine learning enables systems to learn and improve automatically from experience."),
        Document(page_content="Python is widely used in data science due to its rich ecosystem of libraries like NumPy and Pandas."),
        Document(page_content="The Amazon rainforest produces 20% of the world’s oxygen and hosts vast biodiversity."),
        Document(page_content="Albert Einstein proposed the theory of relativity, which changed modern physics."),
        Document(page_content="Neural networks are computational models inspired by the human brain’s structure."),
        Document(page_content="The Sun is approximately 93 million miles away from Earth and is the center of our solar system."),
        Document(page_content="Retrieval-Augmented Generation (RAG) combines retrieval and generation to enhance factual accuracy."),
        Document(page_content="OpenAI developed GPT models that leverage massive datasets to achieve human-like text generation.")
    ]

    # Log summary information for verification
    print(f"Total documents loaded: {len(documents)}")
    print("Displaying sample documents:")
    for i, doc in enumerate(documents[:3]):
        print(f"Document {i+1}: {doc.page_content[:100]}...")

    return documents
    
documents = load_documents()

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

# Display the created chunks
print(f"\nTotal chunks created: {len(chunks)}")
print("Displaying first 2 sample chunks:")
for i, chunk in enumerate(chunks[:2]):
    print(f"Chunk {i+1}: {chunk.page_content}")

Total documents loaded: 10
Displaying sample documents:
Document 1: The Eiffel Tower is located in Paris, France, and stands at 324 meters tall....
Document 2: The Great Wall of China stretches over 13,000 miles and was built to prevent invasions....
Document 3: Machine learning enables systems to learn and improve automatically from experience....

Total chunks created: 10
Displaying first 2 sample chunks:
Chunk 1: The Eiffel Tower is located in Paris, France, and stands at 324 meters tall.
Chunk 2: The Great Wall of China stretches over 13,000 miles and was built to prevent invasions.


In [7]:
# Initialize an empty vector store to hold embeddings for each chunk
vector_store = []

# Convert each text chunk into a dense vector representation (embedding)
for idx, chunk in enumerate(chunks):
    text_content = chunk.page_content
    embedding_vector = model.encode(text_content)   # Convert text → vector
    vector_store.append(embedding_vector)

# Display the first 5 dimensions of first two chunk embeddings (for inspection)
[vector_store[0][:5], vector_store[1][:5]]

[array([ 0.06086046,  0.06121255, -0.00085707, -0.03503641, -0.01374615],
       dtype=float32),
 array([ 0.01918816,  0.10575974,  0.02296079, -0.02900701,  0.03200014],
       dtype=float32)]

# Step 2: Retrieving

This section demonstrates the **Retrieval** stage, where a **query is encoded** into the same embedding space as the indexed data and compared against stored chunk vectors using **cosine similarity**.
The most **semantically aligned chunk** is then identified and retrieved as context for the generative model.
This process replicates the core reasoning mechanism in RAG systems, dynamically grounding responses in relevant, externally indexed information.

In [8]:
def retrieve_most_similar_chunk(model, chunks, vector_store, query_text):
    """
    Given a query, finds the most semantically similar chunk.
    Steps:
    1. Encode the query text into a vector.
    2. Compute cosine similarity with every chunk vector.
    3. Return the chunk with the highest similarity score.
    """
    encoded_query = model.encode(query_text)

    similarity_scores = []
    indexed_scores = []

    # Calculate similarity for each stored embedding
    for idx, chunk_vector in enumerate(vector_store):
        score = calculate_cosine_similarity_score_vect(encoded_query, chunk_vector)
        similarity_scores.append(score)
        indexed_scores.append((score, idx))

    # Log all similarity values for reference
    print("Calculated cosine similarity scores for each chunk:")
    for score, idx in indexed_scores:
        print(f"Score = {score:.4f}  |  Chunk Index = {idx}")

    # Retrieve the chunk with the maximum similarity score
    best_match_index = np.argmax(similarity_scores)
    best_chunk = chunks[best_match_index].page_content
    return best_chunk

In [9]:
# Example query
query = "What is Machine learning"

# Retrieve the most semantically similar chunk to the query
retrieved_chunk = retrieve_most_similar_chunk(model, chunks, vector_store, query)
print("\nMost Relevant Chunk Selected: ", retrieved_chunk)

Calculated cosine similarity scores for each chunk:
Score = -0.0317  |  Chunk Index = 0
Score = -0.0299  |  Chunk Index = 1
Score = 0.6907  |  Chunk Index = 2
Score = 0.2413  |  Chunk Index = 3
Score = 0.0288  |  Chunk Index = 4
Score = 0.1353  |  Chunk Index = 5
Score = 0.3424  |  Chunk Index = 6
Score = 0.0768  |  Chunk Index = 7
Score = 0.1536  |  Chunk Index = 8
Score = 0.1574  |  Chunk Index = 9

Most Relevant Chunk Selected:  Machine learning enables systems to learn and improve automatically from experience.


# Step-3: Generation
This step demonstrates how the system uses the *retrieved contextual information* to generate a coherent, fact-grounded response. The generative model takes both the **user query** and the **retrieved chunks** as input, then formulates a refined output that summarizes and enhances the context.

**Core Idea:**
Generation = *f( Query + Retrieved Context )*
→ Produces a **final synthesized answer** that balances retrieval accuracy with linguistic fluency.

**Typical Components:**

1. **Prompt Construction** – Combine the retrieved documents with clear system instructions.
2. **Model Invocation** – Pass the constructed prompt to the chosen LLM (Gemini, OpenAI, or local model).
3. **Post-Processing** – Optionally summarize, clean, or structure the output for readability and research clarity.

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(model="gemini-2.5-flash")
agent_prompt = """
    You are an intelligent research assistant capable of grounded reasoning.
    Your task is to generate a concise, factual, and contextually accurate answer
    based on the retrieved context provided below.
    
    -------------------------------
    Context:
    {retrieved_chunk}
    -------------------------------
    
    Question:
    {user_query}

    Instructions:
    1. Use the provided retrieved context to answer the user query.
    2. If the context does not contain enough information, clearly state that.
    3. Avoid hallucination or assumptions beyond the provided context.
    4. Respond in clear, structured language suitable for a research explanation.
    5. Do not format the answer in markdown — respond in plain text only.
    6. Summarize the relevant information from the retrieved context before answering.
    7. Enhance the clarity, coherence, and academic tone of the response to ensure it reads like a refined summary, not raw text output.
    """

prompt = ChatPromptTemplate.from_messages([
    ("system", agent_prompt),
    ("user", "{user_query}")
])

chain = (prompt | llm | StrOutputParser())

response = chain.invoke({"retrieved_chunk": retrieved_chunk, "user_query": query})
print(response)

Machine learning is a field that empowers systems to autonomously learn and enhance their performance through accumulated experience.
